In [35]:
import re
from langchain_community.document_loaders import PyPDFium2Loader
from chonkie import TokenChunker
import tiktoken
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.retrievers import TavilySearchAPIRetriever,EnsembleRetriever,ContextualCompressionRetriever
from langchain_community.document_compressors import JinaRerank
from langchain_community.retrievers import BM25Retriever
from langchain_core.runnables import RunnablePassthrough,RunnableLambda,RunnableParallel
from langchain_classic.prompts import ChatPromptTemplate,load_prompt
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.documents import Document
from pinecone import Pinecone, ServerlessSpec
from langchain_ollama import ChatOllama
import uuid
from langchain_community.embeddings import OllamaEmbeddings 
from langchain_pinecone import PineconeVectorStore
import os
load_dotenv()
from youtube_transcript_api import YouTubeTranscriptApi
import warnings
warnings.filterwarnings('ignore')

def extract_video_id(url):
    """
    Extracts the 11-character video ID from various styles of YouTube URLs.
    """
    pattern = r'(?:v=|\/v\/|youtu\.be\/|\/embed\/)([a-zA-Z0-9_-]{11})'
    match = re.search(pattern, url)
    return match.group(1) if match else None

def clean_transcript_text(text):
    """
    Cleans raw automated transcripts by removing filler noise, fixing brackets,
    normalizing whitespace, and correcting basic capitalization/spacing.
    """
    if not text:
        return ""
        
    # 1. Lowercase target tags often injected by whisper/transcribers
    text = text.replace("[Music]", "").replace("[Laughter]", "").replace("(music)", "")
    
    # 2. Remove any remaining text inside square brackets or parentheses e.g. [applause]
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\(.*?\)', '', text)
    
    # 3. Clean up broken spacing around spacing apostrophes (e.g., "you 're" -> "you're")
    text = re.sub(r"\s+'\s*([a-zA-Z])", r"'\1", text)
    
    # 4. Collapse multi-spaces, trailing spaces, tabs, and newline breaks into single spaces
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    # 5. Optional capitalization: Capitalize the first letter of the overall text string
    if len(text) > 0:
        text = text[0].upper() + text[1:]
        
    return text

def get_and_translate_transcript(youtube_url, target_lang='en'):
    video_id = extract_video_id(youtube_url)
    if not video_id:
        return "Error: Invalid YouTube URL format."
    
    try:
        api_client = YouTubeTranscriptApi()
        transcript_list = api_client.list(video_id)
        
        try:
            transcript = transcript_list.find_transcript([target_lang])
            raw_data = transcript.fetch()
        except Exception:
            transcript = transcript_list.find_transcript([]) 
            print(f"🌍 Foreign podcast detected! Original language: '{transcript.language_code}'")
            translated_transcript = transcript.translate(target_lang)
            raw_data = translated_transcript.fetch()
            
        full_text = " ".join([
            entry.text if hasattr(entry, 'text') else entry['text'] 
            for entry in raw_data
        ])
        
        # ─── RUN THE CLEANING MECHANISM HERE ───
        cleaned_text = clean_transcript_text(full_text)
        return cleaned_text

    except Exception as e:
        return f"An error occurred: {str(e)}"

# Example Usage
if __name__ == "__main__":
    # Target URL: Robert Greene on Figuring Out Podcast
    video_url = "https://www.youtube.com/watch?v=p4qPp_3_f2Y" 
    
    cleaned_english_text = get_and_translate_transcript(video_url, target_lang='en')
   
    
gpt4o_tokenizer = tiktoken.get_encoding("cl100k_base") 
split_text = TokenChunker(tokenizer=gpt4o_tokenizer, chunk_size=512, chunk_overlap=50)

from langchain_core.documents import Document

# ... Your prior tokenizer code ...

# Step 2: FIXED LOOP — Extract the text property from the custom Chunk object
langchain_documents = []
chunks = split_text.chunk(cleaned_english_text.lower()) 

for c in chunks:
    # ─── THE CRITICAL FIX ───
    # Pull out the text string attribute from your chunk object
    chunk_string = c.text  
    
    # Optional: Safely capture start/end bounds as metadata if you want to keep them
    chunk_metadata = {
        "start_index": getattr(c, "start_index", None),
        "end_index": getattr(c, "end_index", None)
    }
    
    # Pass the raw string to page_content so Pydantic validation passes successfully
    doc = Document(page_content=chunk_string, metadata=chunk_metadata)
    langchain_documents.append(doc)
    
# Step 3: safely initialize Chroma
embedding_model = JinaEmbeddings(model_name="jina-embeddings-v3", dimensions=1024)
#chrome_db = Chroma.from_documents(documents=langchain_documents, embedding=embedding_model)

# 2. Correctly initialize the base Pinecone client (only pass api_key)
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))

index_name = 'testing-pinecode-emb'

# 3. Verify and create the index with the CORRECT dimensions
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024 ,  # <-- FIXED: 768 exactly matches 'nomic-embed-text'
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )
    
uuids = [str(uuid.uuid4())for i in range(len(langchain_documents))]

pine_db = PineconeVectorStore.from_documents(
    documents=langchain_documents,
    embedding=embedding_model,
    index_name=index_name,
    ids=uuids,
    batch_size=200 # Handles your records safely in batches
)

dense_vector= pine_db.as_retriever(search_type ='mmr', search_kwargs={"k":5})

sparse_vector = BM25Retriever.from_documents(langchain_documents)

tavily_retriever  = TavilySearchAPIRetriever(include_domains=['google.com'],search_depth='advanced')
tavily_retriever .k = 5


hrbrid_retiver = EnsembleRetriever(retrievers=[dense_vector,sparse_vector,tavily_retriever ], weights=[0.4,0.2,0.4])

reranker = JinaRerank(model="jina-reranker-v2-base-multilingual", top_n=5)

compressor = ContextualCompressionRetriever(base_retriever=hrbrid_retiver,base_compressor=reranker)

model = ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite',temperature=0,max_tokens=1000)

text = """You are a precise information retrieval assistant. You must strictly follow the rules below:
    Rule 1: Retrieve information only from the provided Context. If the context does not contain the answer, you must state exactly: "I don't know."
    Rule 2: Do not use any outside knowledge, assumptions, or facts not explicitly written in the Context below.

    Context:
    {context}

    Question:
    {question}

    Answer:"""

prompt = ChatPromptTemplate.from_template(text)
parser = StrOutputParser()

def format_string(doc):
    return '\n\n'.join(d.page_content for d in doc)

Rag_chain = (
    							RunnableParallel({
                            'context':compressor | RunnableLambda(format_string),
                            'question':RunnablePassthrough()
                              })
                              | prompt | model | parser
                               )
result = Rag_chain.invoke("summarize about the youtube podcast in 5 lines number it like 1,2,3,4 in new line ")

print(result)

1. The podcast is viewed as an additional piece or extension of your channel content strategy, functioning as a weekly, recurring piece of content.
2. Video podcasts generally feature a simple execution, such as a wide shot for hosts in a single location or establishing shots for remote production with individual cameras.
3. You can promote the availability of the video podcast within the audio podcast to ensure interested audiences know it exists and where to find it.
4. Your podcast may have an established audience that actively seeks it out, or its audience and your existing channel audience may be different.
